# Add Embeddings Columns to AdventureWorks Silver Tables (Databricks)
This notebook reads these Unity Catalog silver tables and writes versions with an **embedding** column computed using a Databricks Model Serving embedding endpoint (example: **BGE Large**).

Tables covered:
- `product`
- `product_category`
- `product_subcategory`
- `purchase_order_header`
- `purchase_order_detail`
- `sales_order_header`
- `sales_order_detail`

Output strategy:
- Default: write new tables with suffix `_emb` (safe).
- Optional: overwrite in place by setting `OVERWRITE_IN_PLACE = True`.


In [0]:
# 0) Install deps (safe on serverless; affects only this notebook session)
%pip install -U requests
%restart_python


In [0]:
# 1) CONFIG

# Source tables (Unity Catalog)
SRC_CATALOG = "mini_project"
SRC_SCHEMA  = "silver_layer"

# Destination
DEST_CATALOG = SRC_CATALOG
DEST_SCHEMA  = SRC_SCHEMA

# Write behavior
OVERWRITE_IN_PLACE = False            # True = replace the silver tables
DEST_SUFFIX = "" if OVERWRITE_IN_PLACE else "_emb"

# Embedding endpoint
# Put your Databricks Model Serving embedding endpoint name here
EMBEDDING_ENDPOINT_NAME = "databricks-bge-large-en"

# Workspace URL (no trailing slash)
DATABRICKS_HOST = "https://datas-datasurge.cloud.databricks.com"

# Fallback token if notebook context token is not available
DATABRICKS_PAT = "<OPTIONAL_PASTE_PAT_HERE>"

# Batch sizes
EMBED_BATCH_SIZE = 64

# Optional: limit rows per table for a quick test (None = full table)
LIMIT_ROWS = None  # example: 200


In [0]:
# 2) Helper: get a Databricks token for Model Serving
def get_databricks_token():
    # Try to reuse the notebook context token
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except Exception:
        pass

    # Fall back to PAT (if provided)
    if DATABRICKS_PAT and not DATABRICKS_PAT.startswith("<"):
        return DATABRICKS_PAT

    raise ValueError(
        "No Databricks token available. Either run on a context that exposes apiToken(), "
        "or paste a PAT into DATABRICKS_PAT."
    )

TOKEN = get_databricks_token()
print("Got Databricks token for Model Serving.")


In [0]:
# 3) Helper: call the embedding endpoint (REST)
import json
import requests

def _parse_embedding_response(data_json):
    # 1) {"data":[{"embedding":[...]}]}
    if isinstance(data_json, dict) and "data" in data_json and data_json["data"]:
        first = data_json["data"][0]
        if isinstance(first, dict) and "embedding" in first:
            return [d["embedding"] for d in data_json["data"]]

    # 2) {"embeddings":[[...],[...]]}
    if isinstance(data_json, dict) and "embeddings" in data_json and data_json["embeddings"]:
        return data_json["embeddings"]

    # 3) {"predictions":[[...],[...]]}
    if isinstance(data_json, dict) and "predictions" in data_json and data_json["predictions"]:
        return data_json["predictions"]

    raise ValueError(f"Unexpected embedding response format: {data_json}")

def embed_texts(text_list):
    url = f"{DATABRICKS_HOST}/serving-endpoints/{EMBEDDING_ENDPOINT_NAME}/invocations"
    headers = {
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json",
    }

    payload = {"input": text_list}
    resp = requests.post(url, headers=headers, data=json.dumps(payload), timeout=180)

    if resp.status_code >= 400:
        raise RuntimeError(f"Embedding request failed: status={resp.status_code}, body={resp.text}")

    return _parse_embedding_response(resp.json())

# Probe dimension (optional sanity check)
probe = embed_texts(["hello world"])
EMBED_DIM = len(probe[0])
print("Embedding dimension:", EMBED_DIM)


In [0]:
# 4) Table specs (keys + how to build embedding text)
from pyspark.sql import functions as F

def _concat_cols(df, cols):
    exprs = [F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols if c in df.columns]
    return F.concat_ws(" | ", *exprs) if exprs else F.lit("")

def build_product_text(df):
    cols = [
        "name","product_number","color","size","product_line","class","style",
        "safety_stock_level","reorder_point","standard_cost","list_price",
        "days_to_manufacture","weight","size_unit_measure_code","weight_unit_measure_code"
    ]
    return _concat_cols(df, cols)

def build_product_category_text(df):
    cols = ["name","row_guid","modified_date"]
    return _concat_cols(df, cols)

def build_product_subcategory_text(df):
    cols = ["name","product_category_id","row_guid","modified_date"]
    return _concat_cols(df, cols)

def build_purchase_order_header_text(df):
    cols = [
        "purchase_order_id","revision_number","status","employee_id","vendor_id","ship_method_id",
        "order_date","ship_date","subtotal","tax_amt","freight","modified_date"
    ]
    return _concat_cols(df, cols)

def build_purchase_order_detail_text(df):
    cols = [
        "purchase_order_id","purchase_order_detail_id","due_date","order_qty","product_id",
        "unit_price","received_qty","rejected_qty","modified_date"
    ]
    return _concat_cols(df, cols)

def build_sales_order_detail_text(df):
    cols = [
        "sales_order_id","sales_order_detail_id","carrier_tracking_number","order_qty","product_id",
        "special_offer_id","unit_price","unit_price_discount","row_guid","modified_date"
    ]
    return _concat_cols(df, cols)

def build_sales_order_header_text(df):
    cols = [
        "sales_order_id","revision_number","order_date","due_date","ship_date","status","online_order_flag",
        "purchase_order_number","account_number","customer_id","salesperson_id","territory_id",
        "bill_to_address_id","ship_to_address_id","ship_method_id","credit_card_id","credit_card_approval_code",
        "currency_rate_id","subtotal","tax_amt"
    ]
    return _concat_cols(df, cols)

TABLE_SPECS = [
    {"name": "product",               "keys": ["product_id"],               "text_builder": build_product_text},
    {"name": "product_category",      "keys": ["product_category_id"],      "text_builder": build_product_category_text},
    {"name": "product_subcategory",   "keys": ["product_subcategory_id"],   "text_builder": build_product_subcategory_text},
    {"name": "purchase_order_header", "keys": ["purchase_order_id"],        "text_builder": build_purchase_order_header_text},
    {"name": "purchase_order_detail", "keys": ["purchase_order_detail_id"], "text_builder": build_purchase_order_detail_text},
    {"name": "sales_order_header",    "keys": ["sales_order_id"],           "text_builder": build_sales_order_header_text},
    {"name": "sales_order_detail",    "keys": ["sales_order_detail_id"],    "text_builder": build_sales_order_detail_text},
]
print("Configured specs for", len(TABLE_SPECS), "tables.")


In [0]:
# 5) Core function: add embedding column and write table
from pyspark.sql import types as T
from pyspark.sql import functions as F

def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def add_embeddings_to_table(table_name, keys, text_builder):
    src_fqn = f"{SRC_CATALOG}.{SRC_SCHEMA}.{table_name}"
    dest_fqn = f"{DEST_CATALOG}.{DEST_SCHEMA}.{table_name}{DEST_SUFFIX}"

    print("\n==============================")
    print("Source:", src_fqn)
    print("Dest  :", dest_fqn)
    print("Keys  :", keys)

    df = spark.table(src_fqn)

    missing_keys = [k for k in keys if k not in df.columns]
    if missing_keys:
        raise ValueError(f"{src_fqn} is missing key columns: {missing_keys}")

    df2 = df.withColumn("embedding_text", text_builder(df))

    if LIMIT_ROWS is not None:
        df2 = df2.limit(int(LIMIT_ROWS))

    rows = df2.select(*keys, "embedding_text").collect()
    print("Rows to embed:", len(rows))

    out_records = []
    for batch in chunk_list(rows, EMBED_BATCH_SIZE):
        texts = [r["embedding_text"] for r in batch]
        embs = embed_texts(texts)

        for r, emb in zip(batch, embs):
            rec = {k: r[k] for k in keys}
            rec["embedding"] = emb
            out_records.append(rec)

    emb_schema = T.StructType(
        [T.StructField(k, df2.schema[k].dataType, True) for k in keys] +
        [T.StructField("embedding", T.ArrayType(T.FloatType(), containsNull=False), True)]
    )
    emb_df = spark.createDataFrame(out_records, schema=emb_schema)

    # Join embeddings back to original rows (keeps all original columns)
    joined = (
        df2.drop("embedding") if "embedding" in df2.columns else df2
    ).join(emb_df, on=keys, how="left")

    joined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(dest_fqn)

    n_null = joined.filter(F.col("embedding").isNull()).count()
    n_total = joined.count()
    print(f"Wrote {dest_fqn}. Null embeddings: {n_null} / {n_total}")

for spec in TABLE_SPECS:
    add_embeddings_to_table(spec["name"], spec["keys"], spec["text_builder"])


## Done
If `OVERWRITE_IN_PLACE = False`, the new tables are created with suffix `_emb`.
Example: `mini_project.silver_layer.purchase_order_header_emb`.


In [0]:
# 6) Optional demo: simple semantic search in Spark (cosine similarity)
from pyspark.sql import functions as F

DEMO_TABLE = f"{DEST_CATALOG}.{DEST_SCHEMA}.product{DEST_SUFFIX}"

query = "lightweight road bike for racing"
q_emb = embed_texts([query])[0]
q_df = spark.createDataFrame([(q_emb,)], ["q_embedding"])

df_demo = spark.table(DEMO_TABLE).filter(F.col("embedding").isNotNull())

scored = (
    df_demo.crossJoin(q_df)
    .withColumn("score", 
        F.expr("""
            aggregate(zip_with(embedding, q_embedding, (x, y) -> x * y), 0D, (acc, v) -> acc + v) /
            (sqrt(aggregate(transform(embedding, x -> x*x), 0D, (acc, v) -> acc + v)) *
             sqrt(aggregate(transform(q_embedding, x -> x*x), 0D, (acc, v) -> acc + v)))
        """)
    )
    .orderBy(F.col("score").desc())
    .select("product_id", "name", "score")
)

display(scored.limit(20))